In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf

sys.path.append(os.path.dirname("../../genomad"))
from genomad import sequence

2025-12-03 10:37:18.676197: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-03 10:37:18.681596: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 10:37:18.848577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-03 10:37:19.630418: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different compu

In [2]:
FRAGMENT_LENGTH = 6_000
FRAGMENT_LENGTH_RANGE = (4, 6_000)
WORD_SIZE = 4
INPUT_SHAPE = FRAGMENT_LENGTH - WORD_SIZE + 1
MIN_LENGTH = WORD_SIZE
MAX_N_COUNT = 4_000
MAX_N_RATE = 0.6
N_CLASSES = 5

In [5]:
def write_tfrecord(tokenized_seq_data: np.ndarray,
                   class_data: np.ndarray,
                   weight_data: np.ndarray,
                   tfrecords_dir: Path,
                   window_count: int):
        tfrecord_path = str(tfrecords_dir / f"{window_count}.tfrec")
        with tf.io.TFRecordWriter(tfrecord_path) as writer:
            for seq, cls, weight in zip(tokenized_seq_data, class_data, weight_data):
                feature = {
                     "sequence": tf.train.Feature(int64_list=tf.train.Int64List(value=seq)),
                     "class": tf.train.Feature(int64_list=tf.train.Int64List(value=[cls])),
                     "sample_weight": tf.train.Feature(float_list=tf.train.FloatList(value=[weight]))
                     }
                example = tf.train.Example(features=tf.train.Features(feature=feature))
                writer.write(example.SerializeToString())


def generate_empirical_distribution(seqname_path: Path, blacklist_path: Path, fasta_path: Path):
     seqname_list = pd.read_table(seqname_path, sep='\t', header=None)[0].tolist()
     with open(blacklist_path, 'rt') as f: fragments_excluded_from_benchmark = set(line.strip() for line in f if line.strip())
     seqname2len_dict = {seq.accession: len(seq) for seq in sequence.read_fasta(fasta_path, uppercase=True, strip_n=True, compress=True)}
     lengths = [seqname2len_dict[seqname]
                for seqname in seqname_list
                if seqname in seqname2len_dict and seqname not in fragments_excluded_from_benchmark]
     unique_lengths, counts = np.unique(lengths, return_counts=True)
     return np.array([unique_lengths, counts])


def generate_data(
        fasta_paths: list[Path],
        classes_array: np.ndarray,
        empirical_length_distribution: np.ndarray,
        tfrecords_dir: Path,
        single_window: bool = False,
        n_records_per_file: int = 10_000,
    ):
        classes, contigs_per_class = np.unique(classes_array, return_counts=True)
        classes2contigs_per_class = dict(zip(classes, contigs_per_class))
        contig_name_array = []
        contig_id_array = []
        tokenized_seq_data = []
        sample_class_data = []
        sample_weight_data = []
        contig_idx = 0
        window_count = 0
        
        for fasta_path_id, fasta_path in enumerate(fasta_paths):
            if fasta_path_id == 0: max_windows = 50
            else: max_windows = 1 if single_window else None
            for contig_id, seq in enumerate(sequence.read_fasta(fasta_path, strip_n=True)):
                contig_name_array.append(seq.accession)
                for fragment in sequence.seq_windows_length_from_empirical_distribution(seq=seq, empirical_distribution=empirical_length_distribution, max_windows=max_windows):
                    for window_n, seq_window in enumerate(sequence.seq_windows(seq=fragment, length=FRAGMENT_LENGTH, min_length=MIN_LENGTH, max_windows=None)):
                        if window_n > 0 and seq_window.count("N") > MAX_N_RATE * len(seq_window): continue
                        tokenized_seq_window = seq_window.seq_ascii.ljust(FRAGMENT_LENGTH, b"N")
                        tokenized_seq_window = sequence.tokenize_dna(seq=tokenized_seq_window, word_size=WORD_SIZE)
                        tokenized_seq_data.append(tokenized_seq_window)
                        sample_class_data.append(int(classes_array[contig_idx]))
                        sample_weight_data.append(float(1_000_000 / int(classes2contigs_per_class[classes_array[contig_idx]]) * (len(seq_window) / len(seq))))
                        contig_id_array.append(contig_idx)
                        window_count += 1
                        if window_count % n_records_per_file == 0:
                            write_tfrecord(tokenized_seq_data=tokenized_seq_data,
                                           class_data=sample_class_data,
                                           weight_data=sample_weight_data,
                                           tfrecords_dir=tfrecords_dir,
                                           window_count=window_count)
                            tokenized_seq_data = []
                            sample_class_data = []
                            sample_weight_data = []
                contig_idx += 1

        if len(tokenized_seq_data):
            write_tfrecord(tokenized_seq_data=tokenized_seq_data,
                           class_data=sample_class_data,
                           weight_data=sample_weight_data,
                           tfrecords_dir=tfrecords_dir,
                           window_count=window_count)

        return np.array(contig_name_array), np.array(contig_id_array)

In [6]:
empirical_length_distribution = generate_empirical_distribution(seqname_path = Path("/data/genomad_supplementary_data_code/benchmark_data/sequence_weights.tsv"),
                                                                blacklist_path = Path("/media/xuzhen/Elements/genomad_supplementary_data_code/benchmark_data/excluded_from_benchmark.txt"),
                                                                fasta_path = Path("/data/genomad_supplementary_data_code/benchmark_data/train_test_sequences.fna.gz"))
tfrecords_dir = Path("/data/DeepMicroClass/data/tfrecords_length_IMG_M/train")
os.makedirs(tfrecords_dir, exist_ok=True)

contigs_classes_windows = pd.read_csv('/data/DeepMicroClass/data/train_class_5_weight.csv', sep=',', header=0)
contigs = contigs_classes_windows['contigs']
classes = contigs_classes_windows['classes'].values
windows_per_contig = contigs_classes_windows['windows_per_contig'].values

generate_data(fasta_paths=[Path("/data/DeepMicroClass/data/clean_sequence/pre20_prokaryote.fa_Train.gz"),
                           Path("/data/DeepMicroClass/data/virus-host-db/prok_vir_Train.fasta.gz"),
                           Path("/data/DeepMicroClass/data/plasmid/pre20_plasmid.fa_Train.fasta.gz"),
                           Path("/data/DeepMicroClass/data/clean_sequence/pre20_eukaryote_Train.fasta.gz"),
                           Path("/data/DeepMicroClass/data/virus-host-db/euk_vir_Train.fasta.gz")],
            classes_array=classes,
            empirical_length_distribution=empirical_length_distribution,
            tfrecords_dir=tfrecords_dir,
            single_window=False,
            n_records_per_file=20_000)


tfrecords_dir = Path("/data/DeepMicroClass/data/tfrecords_length_IMG_M/validate")
os.makedirs(tfrecords_dir, exist_ok=True)

contigs_classes_windows = pd.read_csv('/data/DeepMicroClass/data/validate_class_5_weight.csv', sep=',', header=0)
contigs = contigs_classes_windows['contigs']
classes = contigs_classes_windows['classes'].values
windows_per_contig = contigs_classes_windows['windows_per_contig'].values

generate_data(fasta_paths=[Path("/data/DeepMicroClass/data/clean_sequence/pre20_prokaryote.fa_Validate.gz"),
                           Path("/data/DeepMicroClass/data/virus-host-db/prok_vir_Validate.fasta.gz"),
                           Path("/data/DeepMicroClass/data/plasmid/pre20_plasmid.fa_Validate.fasta.gz"),
                           Path("/data/DeepMicroClass/data/clean_sequence/pre20_eukaryote_Validate.fasta.gz"),
                           Path("/data/DeepMicroClass/data/virus-host-db/euk_vir_Validate.fasta.gz")],
            classes_array=classes,
            empirical_length_distribution=empirical_length_distribution,
            tfrecords_dir=tfrecords_dir,
            single_window=False,
            n_records_per_file=20_000)

(array(['CP003929.1', 'CP003372.1', 'AHJR01000001.1', ..., 'NC_055567',
        'NC_022006', 'NC_020103'], shape=(14814,), dtype='<U15'),
 array([    0,     0,     0, ..., 14812, 14812, 14813], shape=(469208,)))